# Gap-fill meteo variables

</br>

# Settings

## Data settings

In [ ]:
SITE_LAT = 47.478333   # CH-LAE
SITE_LON = 8.364389  # CH-LAE
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # Timezone, e.g. "1" is translated to timezone "UTC+01:00" (CET, winter time)

## Imports

In [ ]:
from datetime import datetime
import importlib.metadata
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
sns.set_theme('notebook')
import diive as dv
from diive.core.plotting.timeseries import TimeSeries
from diive.core.io.files import load_parquet, save_parquet
from diive.core.plotting.heatmap_datetime import HeatmapDateTime
from diive.core.times.times import DetectFrequency
from diive.core.times.times import TimestampSanitizer
from diive.pkgs.createvar.potentialradiation import potrad
from diive.pkgs.corrections.offsetcorrection import remove_relativehumidity_offset, remove_radiation_zero_offset
from diive.pkgs.gapfilling.longterm import LongTermGapFillingXGBoostTS
import warnings
from influxdb_client.client.warnings import MissingPivotFunction
warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
version_diive = importlib.metadata.version("diive")
print(f"diive version: v{version_diive}")

</br>

# Load data from files

In [ ]:
filename_meteo6_2004_2025 = "02_METEO6_NOT-GAPFILLED_2004-2025.parquet"
meteo6_2004_2025 = load_parquet(filepath=filename_meteo6_2004_2025)
meteo6_2004_2025

In [ ]:
# meteo6_2004_2025 = meteo6_2004_2025.loc[meteo6_2004_2025.index.year > 2020].copy()

In [ ]:
meteo6_2004_2025.describe()

</br>

# Gap-filling

## SW_IN_T1_47_1

### Target and features

In [ ]:
TARGET = "SW_IN_T1_47_1"
subset = pd.DataFrame()
subset[TARGET] = meteo6_2004_2025[[TARGET]].copy()
subset['SW_IN_POT'] = potrad(timestamp_index=subset.index, lat=SITE_LAT, lon=SITE_LON, utc_offset=1)
subset

### Initialize gap-filling

In [ ]:
ltxgb = LongTermGapFillingXGBoostTS(
    input_df=subset,
    target_col=TARGET,
    verbose=0,
    # features_lag=[-1, -1],
    vectorize_timestamps=True,
    add_continuous_record_number=True,
    sanitize_timestamp=True,
    perm_n_repeats=9,
    n_estimators=1000,
    random_state=42,
    # booster='gbtree',  # gbtree (default), gblinear, dart
    # device='cpu',
    validate_parameters=True,
    # disable_default_eval_metric=False,
    early_stopping_rounds=50,
    max_depth=6,
    # max_delta_step=0,
    # subsample=1,
    learning_rate=0.3,
    # min_split_loss=0,
    # min_child_weight=1,
    # colsample_bytree=1,
    # colsample_bylevel=1,
    # colsample_bynode=1,
    # reg_lambda=1,
    # reg_alpha=0,
    tree_method='auto',  # auto, hist, approx, exact
    # scale_pos_weight=1,
    # grow_policy=0,
    grow_policy='lossguide',  # depthwise, lossguide
    # max_leaves=0,
    # max_bin=256,
    # num_parallel_tree=1,
    n_jobs=-1
)

### Initialize yearly models

In [ ]:
ltxgb.create_yearpools()
ltxgb.initialize_yearly_models()

### Reduce features across years

In [ ]:
ltxgb.reduce_features_across_years()

### Train models and fill gaps

In [ ]:
ltxgb.fillgaps()

### Results: yearly scores

In [ ]:
scores = pd.DataFrame(ltxgb.scores_)
scores.style.format("{:.4f}")

### Results: feature importances

In [ ]:
ltxgb.feature_importances_yearly_

In [ ]:
ltxgb.showplot_feature_ranks_per_year()

### Results: plot gap-filled time series

In [ ]:
observed = subset[TARGET]
gapfilled = ltxgb.gapfilled_
locs_filled = observed.isna()
gapvalues = gapfilled[locs_filled].copy()
# gapfilled[locs_filled].describe()
_plotdf = pd.DataFrame({
    'observed': observed,
    'gapfilled': gapfilled,
    'gapvalues': gapvalues
})
_plotdf.plot(subplots=True, x_compat=True);

### Add gap-filled series to dataframe

In [ ]:
meteo6_2004_2025[gapfilled.name] = gapfilled

### Plot time series

In [ ]:
meteo6_2004_2025[[TARGET, gapfilled.name]].plot(subplots=True, x_compat=True);

### Correction: Remove zero offset < 0 from `SW_IN_T1_47_1_gfXG`
- Additional correction needed because gap-filling introduced values < 0

In [ ]:
meteo6_2004_2025[[TARGET, gapfilled.name]].describe()

In [ ]:
_swin = meteo6_2004_2025['SW_IN_T1_47_1_gfXG'].copy()
_swin_corrected = remove_radiation_zero_offset(series=_swin, lat=SITE_LAT, lon=SITE_LON, utc_offset=1, showplot=True)
meteo6_2004_2025['SW_IN_T1_47_1_gfXG'] = np.nan
meteo6_2004_2025['SW_IN_T1_47_1_gfXG'] = _swin_corrected

### Final dataframe

In [ ]:
meteo6_2004_2025

In [ ]:
meteo6_2004_2025.describe()

</br>

</br>

</br>

## TA_T1_47_1

### Target and features

In [ ]:
TARGET = "TA_T1_47_1"
subset = pd.DataFrame()
subset[TARGET] = meteo6_2004_2025[[TARGET]].copy()
subset['SW_IN_T1_47_1_gfXG'] = meteo6_2004_2025['SW_IN_T1_47_1_gfXG'].copy()
# subset['SW_IN_POT'] = potrad(timestamp_index=subset.index, lat=SITE_LAT, lon=SITE_LON, utc_offset=1)
subset

### Initialize gap-filling

In [ ]:
ltxgb = LongTermGapFillingXGBoostTS(
    input_df=subset,
    target_col=TARGET,
    verbose=0,
    # features_lag=[-1, -1],
    vectorize_timestamps=True,
    add_continuous_record_number=True,
    sanitize_timestamp=True,
    perm_n_repeats=9,
    n_estimators=1000,
    random_state=42,
    # booster='gbtree',  # gbtree (default), gblinear, dart
    # device='cpu',
    validate_parameters=True,
    # disable_default_eval_metric=False,
    early_stopping_rounds=50,
    max_depth=6,
    # max_delta_step=0,
    # subsample=1,
    learning_rate=0.3,
    # min_split_loss=0,
    # min_child_weight=1,
    # colsample_bytree=1,
    # colsample_bylevel=1,
    # colsample_bynode=1,
    # reg_lambda=1,
    # reg_alpha=0,
    tree_method='auto',  # auto, hist, approx, exact
    # scale_pos_weight=1,
    # grow_policy=0,
    grow_policy='lossguide',  # depthwise, lossguide
    # max_leaves=0,
    # max_bin=256,
    # num_parallel_tree=1,
    n_jobs=-1
)

### Initialize yearly models

In [ ]:
ltxgb.create_yearpools()
ltxgb.initialize_yearly_models()

### Reduce features across years

In [ ]:
ltxgb.reduce_features_across_years()

### Train models and fill gaps

In [ ]:
ltxgb.fillgaps()

### Results: yearly scores

In [ ]:
scores = pd.DataFrame(ltxgb.scores_)
scores.style.format("{:.4f}")

### Results: feature importances

In [ ]:
ltxgb.feature_importances_yearly_

In [ ]:
ltxgb.showplot_feature_ranks_per_year()

### Results: plot gap-filled time series

In [ ]:
observed = subset[TARGET]
gapfilled = ltxgb.gapfilled_
locs_filled = observed.isna()
gapvalues = gapfilled[locs_filled].copy()
# gapfilled[locs_filled].describe()
_plotdf = pd.DataFrame({
    'observed': observed,
    'gapfilled': gapfilled,
    'gapvalues': gapvalues
})
_plotdf.plot(subplots=True, x_compat=True);

### Add gap-filled series to dataframe

In [ ]:
meteo6_2004_2025[gapfilled.name] = gapfilled

### Plot time series

In [ ]:
meteo6_2004_2025[[TARGET, gapfilled.name]].plot(subplots=True, x_compat=True);

### Final dataframe

In [ ]:
meteo6_2004_2025

In [ ]:
meteo6_2004_2025.describe()

</br>

</br>

</br>

## PPFD_IN_T1_47_1

### Target and features

In [ ]:
TARGET = "PPFD_IN_T1_47_1"
subset = pd.DataFrame()
subset[TARGET] = meteo6_2004_2025[[TARGET]].copy()
subset['SW_IN_T1_47_1_gfXG'] = meteo6_2004_2025['SW_IN_T1_47_1_gfXG'].copy()
# subset['SW_IN_POT'] = potrad(timestamp_index=subset.index, lat=SITE_LAT, lon=SITE_LON, utc_offset=1)
subset

### Initialize gap-filling

In [ ]:
ltxgb = LongTermGapFillingXGBoostTS(
    input_df=subset,
    target_col=TARGET,
    verbose=0,
    # features_lag=[-1, -1],
    vectorize_timestamps=True,
    add_continuous_record_number=True,
    sanitize_timestamp=True,
    perm_n_repeats=9,
    n_estimators=1000,
    random_state=42,
    # booster='gbtree',  # gbtree (default), gblinear, dart
    # device='cpu',
    validate_parameters=True,
    # disable_default_eval_metric=False,
    early_stopping_rounds=50,
    max_depth=6,
    # max_delta_step=0,
    # subsample=1,
    learning_rate=0.3,
    # min_split_loss=0,
    # min_child_weight=1,
    # colsample_bytree=1,
    # colsample_bylevel=1,
    # colsample_bynode=1,
    # reg_lambda=1,
    # reg_alpha=0,
    tree_method='auto',  # auto, hist, approx, exact
    # scale_pos_weight=1,
    # grow_policy=0,
    grow_policy='lossguide',  # depthwise, lossguide
    # max_leaves=0,
    # max_bin=256,
    # num_parallel_tree=1,
    n_jobs=-1
)

### Initialize yearly models

In [ ]:
ltxgb.create_yearpools()
ltxgb.initialize_yearly_models()

### Reduce features across years

In [ ]:
ltxgb.reduce_features_across_years()

### Train models and fill gaps

In [ ]:
ltxgb.fillgaps()

### Results: yearly scores

In [ ]:
scores = pd.DataFrame(ltxgb.scores_)
scores.style.format("{:.4f}")

### Results: feature importances

In [ ]:
ltxgb.feature_importances_yearly_

In [ ]:
ltxgb.showplot_feature_ranks_per_year()

### Results: plot gap-filled time series

In [ ]:
observed = subset[TARGET]
gapfilled = ltxgb.gapfilled_
locs_filled = observed.isna()
gapvalues = gapfilled[locs_filled].copy()
# gapfilled[locs_filled].describe()
_plotdf = pd.DataFrame({
    'observed': observed,
    'gapfilled': gapfilled,
    'gapvalues': gapvalues
})
_plotdf.plot(subplots=True, x_compat=True);

### Add gap-filled series to dataframe

In [ ]:
meteo6_2004_2025[gapfilled.name] = gapfilled

### Plot time series

In [ ]:
meteo6_2004_2025[[TARGET, gapfilled.name]].plot(subplots=True, x_compat=True);

### Correction: Remove zero offset < 0 from `PPFD_IN_T1_47_1_gfXG`
- Additional correction needed because gap-filling introduced values < 0

In [ ]:
meteo6_2004_2025[[TARGET, gapfilled.name]].describe()

In [ ]:
_ppfdin = meteo6_2004_2025['PPFD_IN_T1_47_1_gfXG'].copy()
_ppfdin_corrected = remove_radiation_zero_offset(series=_ppfdin, lat=SITE_LAT, lon=SITE_LON, utc_offset=1, showplot=True)
meteo6_2004_2025['PPFD_IN_T1_47_1_gfXG'] = np.nan
meteo6_2004_2025['PPFD_IN_T1_47_1_gfXG'] = _ppfdin_corrected

### Final dataframe

In [ ]:
meteo6_2004_2025

In [ ]:
meteo6_2004_2025.describe()

</br>

</br>

</br>

</br>

# Plot time series

In [ ]:
_plot_df = meteo6_2004_2025.copy()
_plot_df = _plot_df.replace(-9999, np.nan)
_plot_df.plot(subplots=True, figsize=(20, 9), title="Gap-filled (some) meteo data", alpha=.9, x_compat=True);

# Plot heatmaps

In [ ]:
meteo6_2004_2025

In [ ]:
fig, axs = plt.subplots(ncols=5, figsize=(24, 10), dpi=100, layout="constrained")
fig.suptitle(f'Half-hourly', fontsize=16)
dv.heatmapdatetime(series=meteo6_2004_2025['PPFD_IN_T1_47_1'], title="PPFD_IN_T1_47_1", ax=axs[0], cb_digits_after_comma=0, zlabel="value").plot()
dv.heatmapdatetime(series=meteo6_2004_2025['PPFD_IN_T1_47_1_gfXG'], title="PPFD_IN_T1_47_1_gfXG", ax=axs[1], cb_digits_after_comma=0, zlabel="value").plot()
dv.heatmapdatetime(series=meteo6_2004_2025['RH_T1_47_1'], title="RH_T1_47_1", ax=axs[2], cb_digits_after_comma=0, zlabel="value").plot()
dv.heatmapdatetime(series=meteo6_2004_2025['LW_IN_T1_47_1'], title="LW_IN_T1_47_1", ax=axs[3], cb_digits_after_comma=0, zlabel="value").plot()
dv.heatmapdatetime(series=meteo6_2004_2025['PA_T1_47_1'], title="PA_T1_47_1", ax=axs[4], cb_digits_after_comma=0, zlabel="value").plot()

In [ ]:
fig, axs = plt.subplots(ncols=4, figsize=(18, 10), dpi=100, layout="constrained")
fig.suptitle(f'Half-hourly', fontsize=16)
dv.heatmapdatetime(series=meteo6_2004_2025['SW_IN_T1_47_1'], title="SW_IN_T1_47_1", ax=axs[0], cb_digits_after_comma=0, zlabel="value").plot()
dv.heatmapdatetime(series=meteo6_2004_2025['SW_IN_T1_47_1_gfXG'], title="SW_IN_T1_47_1_gfXG", ax=axs[1], cb_digits_after_comma=0, zlabel="value").plot()
dv.heatmapdatetime(series=meteo6_2004_2025['TA_T1_47_1'], title="TA_T1_47_1", ax=axs[2], cb_digits_after_comma=0, zlabel="value").plot()
dv.heatmapdatetime(series=meteo6_2004_2025['TA_T1_47_1_gfXG'], title="TA_T1_47_1_gfXG", ax=axs[3], cb_digits_after_comma=0, zlabel="value").plot()

</br>

# Stats

In [ ]:
meteo6_2004_2025.describe()

</br>

# Save to file

In [ ]:
OUTNAME = "12_METEO6_GAPFILLED_2004-2025"
OUTPATH = r""
filepath = save_parquet(filename=OUTNAME, data=meteo6_2004_2025, outpath=OUTPATH)
meteo6_2004_2025.to_csv(Path(OUTPATH) / f"{OUTNAME}.csv")

</br>

# End of notebook.

In [ ]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")

</br>